## Agent Behavior
### 行为笔记

单个健康机器人每一步的行为                       

  1. 故障检测  faulty_check()                                 
     扫描 detection_radius 内邻居 ──发现可疑──▶ 变身 Granuloma  
                                                              
  2. 意见更新  update_opinion()   (三选一策略)                 
     · 观测脚下地面（安全/不安全）                              
     · 用证据更新自己的 opinion（贝叶斯）                       
     · 与邻居汇聚意见（排除 Granuloma 邻居）                    
                                                              
  3. 移动  update_velocity() → update_position()               
     · 相关随机游走（直行+随机转向）                            
     · 靠近边界时反弹                                          
                                                              
 全场 ≥90% 健康机器人意见一致 → 达成共识，仿真结束           

### 机器人类型

Healthy     正常移动、观测、更新意见
Faulty      意见永远卡在 0.05，散布错误信息
Granuloma   健康机器人发现疑似故障者后转变而来，专门去围堵追踪某个故障机器人

### 1.single time step

In [ ]:
def step(self):
    self.granuloma_agents.double_faulty_check()      # ① 追踪者复核目标是否真故障
    self.healthy_agents.faulty_check()               # ② 健康机器人扫描周围找故障者
    self.define_subsets()                            # 重新划分三类子集
    self.non_dupe()                                  # 去重：避免两个追踪者盯同一目标
    self.healthy_agents.update_opinion()             # ③ 健康机器人更新意见
    self.non_granuloma_agents.update_velocity()      # ④ 普通机器人更新速度（移动）
    self.granuloma_agents.update_tracking_velocity() # ⑤ 追踪者朝目标移动
    self.granuloma_agents.update_tracking_time()
    self.agents.update_position()                    # ⑥ 所有机器人根据速度移动

### 2.Movement

random walk

转向时机由指数分布控制（time_period=50），所以是一种相关随机游走（correlated random walk）——走一阵直线再随机转，比纯布朗运动更接近真实机器人。

In [ ]:
def update_vel(agent):
    old_vel = agent.vel
    if turn(agent):                                  # 直行时间到了就转向
        current_direction = get_direction(old_vel)
        turning_amount = generate_turning_amount()   # 随机转 0~2π
        new_direction = current_direction + turning_amount
        magnitude = np.linalg.norm(agent.vel)
        new_vel = np.array([magnitude*sin(new_direction),
                            magnitude*cos(new_direction)])
        agent.time_before_turn = int(agent.model.nprandom.exponential(agent.p.time_period))  # 下次转向间隔服从指数分布
        agent.time_straight = 0
        return new_vel
    else:
        agent.time_straight += 1                     # 否则继续直行
        return old_vel

### 3.Avoid Border

In [ ]:
def avoid_borders(self):
    for dim in range(self.p.ndim):
        if self.pos[dim] < int(self.p.size/20):                 # 离左/下边界太近
            self.vel[dim] += self.p.speed*0.5
        if self.pos[dim] > self.p.size - int(self.p.size/20):   # 离右/上边界太近
            self.vel[dim] -= self.p.speed*0.5
    self.vel = normalise(self.vel)

#安全补丁
每个策略都先运行这个安全补丁

search_space是一个size*size的numpy数组，size=200，所以索引：0～199
机器人agent.pos是agentpy.Space里的连续浮点坐标，范围是[0,200]
观测时要把连续坐标转换为数组下标：
safe = not agent.model.search_space[int(agent.pos[0]), int(agent.pos[1])]
当机器人在右边/上边边界时，pos[i]为200，即
int(200.0) == 200   # 但数组只有 0~199，索引 200 越界 → IndexError
该代码专门用于防止该情况，
Space 把坐标钳制在 [0, 200]，触发条件时 pos[i] 实际就是 200.0。减掉 0.5 后变成 199.5：
int(199.5) == 199   # 合法，落在最后一格的中间

注：只要减去的值在（0，1）之间，都能把索引拉回199，减去0.5是让它落在最后一格的中间位置

In [ ]:
def position_check(agent):
    # Check agents position doesn't round up to an index out of range
    for i in range(agent.p.ndim):
        if int(agent.pos[i]) == agent.p.size:
            agent.pos[i] -= 0.5
    return agentupdate opinion

### 4.update opinion

In [ ]:
#strategy01: pooling (SProdOP)

def update_opinion(agent):
    agent = collect_evidence(agent)                  # 每步采样脚下地面：安全 +1，不安全 -1
    if agent.model.t % agent.p.tau_evidence == 0:    # 每 30 步：用累积证据贝叶斯式更新自己意见
        agent = evidentially_update(agent)
        agent = bounds_check(agent)                  # 意见夹紧到 [0.05, 0.95]
        agent.obs_count = 0
    if agent.model.t % agent.p.tau_sharing == 0:     # 每 20 步：和邻居"汇聚"意见
        inner_nbs = agent.neighbors(agent, distance=agent.p.detection_radius)
        agent = pool_opinions(agent, inner_nbs)
        agent = bounds_check(agent)
    return agent

#自我证据更新：根据观测把意见往 0 或 1 推（贝叶斯似然比，alpha=0.2 是观测噪声）
def evidentially_update(agent):
    if agent.obs_count > 0:
        agent.opinion = ((1-agent.p.alpha)*agent.opinion)/(((1-agent.p.alpha)*agent.opinion)+agent.p.alpha*(1-agent.opinion))
    elif agent.obs_count < 0:
        agent.opinion = (agent.p.alpha*agent.opinion)/((agent.p.alpha*agent.opinion)+((1-agent.p.alpha)*(1-agent.opinion)))
    else:
        pass
        
    return agent

#意见汇聚 乘积型SProdOp用对数意见池把自己和邻居的意见相乘归一化，汇聚前会检查邻居是否被当作faulty被跟踪，如果是的话就不采用其意见 

def pool_opinions(agent, nbs):
    # Pool if agent has neighbours and none are granuloma
    if len(nbs) > 0 and "Granuloma" not in nbs.type:
        nbs_ops_array = np.array(nbs.opinion)
        pool_array = np.append(nbs_ops_array, agent.opinion)    # add agent's opinion back in
        h1 = (np.prod(pool_array))**agent.p.w
        h2 = (np.prod(1-pool_array))**agent.p.w
        # Update opinion using SProdOp
        agent.opinion = h1/(h1+h2)
    return agent

In [ ]:
#strategy02: BBots(bayes_bots)
#单个机器人累计两类观测计数 observations=[安全数, 不安全数]，用 Beta 分布后验判断：

def cdf_check(agent):
    p = beta.cdf(0.5, agent.observations[0] + agent.p.alpha_0,
                       agent.observations[1] + agent.p.alpha_0)
    if p > agent.p.p_c:        # 后验超过置信阈值 0.9 → 决定 H2
        agent.opinion = 0.05
    elif (1-p) > agent.p.p_c:  # → 决定 H1
        agent.opinion = 0.95
    return agent

#每 tau_evidence=30 步亲自观测一次脚下
def update_opinion(agent):
    """
    Update agent's opinion using the 'Bayes Bots' algorithm
    """
    # Make an observation every tau seconds
    if agent.model.t % agent.p.tau_evidence == 0:
        agent = make_observation(agent)
    # If undecided, get the neighbouring agents last observations and update posterior
    if agent.opinion == 0.5 and agent.model.t % agent.p.tau_sharing == 0:
        # See what the neighbours are saying
        agent = neighbour_time(agent)
        # Try and make a decision
        agent = cdf_check(agent)

    return agent


#未决定时（opinion==0.5）还会收集邻居的最近观测补充进自己的证据计数——u_plus=True 时直接采纳邻居"已决定的意见"，否则只采纳邻居的"最近一次原始观测"。
def neighbour_time(agent):
    nbs = agent.neighbors(agent, distance=agent.p.detection_radius)
    if "Granuloma" not in nbs.type:
        for nb in nbs:
            if agent.p.u_plus == True and nb.opinion != 0.5:        # If neighbour is 'decided' take that as their opinion
                # See what the neighbours are saying
                if nb.opinion == 0.95:
                    agent.observations[0] += 1
                elif nb.opinion == 0.05:
                    agent.observations[1] += 1
            else:                                                   # If not get what their last observation was
                # See what the neighbours are saying
                if nb.last_observation == 0.95:
                    agent.observations[0] += 1
                elif nb.last_observation == 0.05:
                    agent.observations[1] += 1
    return agent

In [ ]:
#strategy03: DMMD
#每个机器人都先collect，然后在explore和disseminate间切换

def update_opinion(agent):
    agent.time_in_state += 1
    if agent.state == "exploration":
        agent = explore(agent)        # 探索：自己采样环境收集证据
    elif agent.state == "dissemination":
        agent = disseminate(agent)    # 传播：广播意见 + 听邻居意见投票
    return agent

### 5.track faulty agents

In [ ]:
def faulty_check(self):
    if self.model.t % self.p.faulty_search_rate == 0:
        nbs = self.neighbors(self, distance=self.p.detection_radius)
        if "Granuloma" not in nbs.type and len(nbs) > 0:
            for nb in nbs:
                # 误报：把健康者当成故障（概率 = false_positive）
                if self.model.random.random() <= self.p.false_positive and nb.type == "Healthy":
                    self.type = "Granuloma"; self.tracking_id = nb.id; self.tracking_time = 1
                # 正报：把真故障者识别出来（概率 = true_positive）
                if nb.type == "Faulty":
                    if self.model.random.random() <= self.p.true_positive:
                        self.type = "Granuloma"; self.tracking_id = nb.id; self.tracking_time = 1

#机器人发现可疑邻居后转变为 Granuloma，记下目标 ID，从此停止正常移动，改为追着目标跑,把它"围住"隔离，同时它的意见在汇聚时被其它机器人排除。
    def update_tracking_velocity(self):
        """
        Granuloma agents update their tracking velocity
        """
        # Get faulty agent to be tracked
        faulty_agent = self.model.agents.select(self.model.agents.id == self.tracking_id)[0]
        self.vel = faulty_agent.pos - self.pos
        # Avoid borders and normalise speed
        self.avoid_borders()
        self.vel *= self.p.speed

#复核 追踪者每隔 double_check_rate 步复核一次。追踪时间越长，识别越准
def double_faulty_check(self):
        """
        Double check that the faulty agent you're tracking is faulty 
        """
        if self.tracking_time % self.p.double_check_rate == 0:
            # Check the actual status of the agent
            actual_status = self.model.agents.select(self.model.agents.id == self.tracking_id)[0].type
            if actual_status == "Healthy":
                if self.model.random.random() <= (1-get_false_positive(self.p.false_positive, self.tracking_time, self.p.mu)):
                    self.type = "Healthy"
                    self.tracking_id = None
                    self.tracking_time = 0
                    self.vel = -self.vel
            else:
                if self.model.random.random() <= (1-get_true_positive(self.p.true_positive, self.tracking_time, self.p.mu)):
                    self.type = "Healthy"
                    self.tracking_id = None
                    self.tracking_time = 0
                    self.vel = -self.vel

#去重 如果两个追踪者盯着同一目标 其中一个释放避免浪费
def non_dupe(self):
        for agent_i in self.granuloma_agents:
            for agent_ii in self.granuloma_agents:
                if agent_i.id != agent_ii.id:
                    if agent_i.tracking_id == agent_ii.tracking_id:
                        agent_i.type = "Healthy"
                        agent_i.tracking_id = None
                        agent_i.tracking_time = 0
        self.define_subsets()

### 6.单个机器人初始化

[agent.py:79-96]：每个机器人出生时随机分配速度方向，并按所选策略初始化。